## TargetLayer 实例化调试

这个 notebook 只验证 `TargetLayer` 的实例化。若相邻层位顺序校验报错，会自动读取当前运行生成的调试 CSV，并可视化违例点在工区中的分布。


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_interpretation_petrel
from cup.seismic.process import TargetLayer, interpolate_interpretation_surface
from cup.seismic.survey import open_survey

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True


In [ ]:
data_root = repo_root / "data"
debug_dir = data_root / "_debug" / "target_layer_instantiation@20260414"
debug_dir.mkdir(parents=True, exist_ok=True)

seismic_file = data_root / "raw" / "mero se 0116_1ms_new_84_coord.Sgy"
segy_iline = 5
segy_xline = 21
segy_istep = 1
segy_xstep = 4

horizon_files = {
    "bve_top": data_root / "interpre_time" / "bve_top_t",
    "bve_bot": data_root / "interpre_time" / "bve_bot_t",
    "itp_bot": data_root / "interpre_time" / "itp_bot_t",
}

for path in [seismic_file, *horizon_files.values()]:
    if not path.exists():
        raise FileNotFoundError(path)

debug_dir


In [ ]:
def build_interpolated_horizons(geometry: dict) -> dict[str, pd.DataFrame]:
    interpolated = {}
    for horizon_name, horizon_file in horizon_files.items():
        raw_df = import_interpretation_petrel(horizon_file)
        interp_df = interpolate_interpretation_surface(
            interpretation_df=raw_df,
            geometry=geometry,
            outlier_threshold=0.02,
            min_neighbor_count=2,
            keep_nan=True,
        )
        interpolated[horizon_name] = interp_df
    return interpolated


def instantiate_target_layer(geometry: dict):
    existing_logs = set(debug_dir.glob("horizon_order_violation__*.csv"))
    interpolated = build_interpolated_horizons(geometry)

    try:
        target_layer = TargetLayer(
            interpolated_horizon_dfs=interpolated,
            geometry=geometry,
            horizon_names=list(horizon_files.keys()),
            debug_dir=debug_dir,
        )
        return target_layer, None, None
    except AssertionError as exc:
        all_logs = set(debug_dir.glob("horizon_order_violation__*.csv"))
        new_logs = sorted(all_logs - existing_logs, key=lambda path: path.stat().st_mtime, reverse=True)
        if new_logs:
            log_path = new_logs[0]
        else:
            historical_logs = sorted(all_logs, key=lambda path: path.stat().st_mtime, reverse=True)
            log_path = historical_logs[0] if historical_logs else None

        violation_df = pd.read_csv(log_path) if log_path is not None else None
        return None, exc, {"log_path": log_path, "violation_df": violation_df}


In [ ]:
survey = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": segy_iline,
        "xline": segy_xline,
        "istep": segy_istep,
        "xstep": segy_xstep,
    },
)
geometry = survey.query_geometry(domain="time")

target_layer, target_layer_error, debug_payload = instantiate_target_layer(geometry)
violation_df = None if debug_payload is None else debug_payload["violation_df"]
violation_log_path = None if debug_payload is None else debug_payload["log_path"]

if target_layer is not None:
    print("TargetLayer instantiated successfully.")
    print("Horizon names:", target_layer.horizon_names)
    print("Zones:", target_layer.iter_zones())
else:
    print(target_layer_error)
    print("Debug log:", violation_log_path)
    if violation_df is not None:
        print("Violation count:", len(violation_df))
        display(violation_df.head(20))


In [ ]:
if violation_df is None or violation_df.empty:
    print("No violation points to visualize.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    scatter = axes[0].scatter(
        violation_df["inline"],
        violation_df["xline"],
        c=violation_df["interpretation_delta"],
        cmap="coolwarm",
        s=28,
        alpha=0.85,
        edgecolors="none",
    )
    axes[0].set_title("Violation Points in Survey")
    axes[0].set_xlabel("inline")
    axes[0].set_ylabel("xline")
    axes[0].set_xlim(float(geometry["inline_min"]), float(geometry["inline_max"]))
    axes[0].set_ylim(float(geometry["xline_min"]), float(geometry["xline_max"]))
    colorbar = fig.colorbar(scatter, ax=axes[0])
    colorbar.set_label("top - bottom")

    axes[1].hist(violation_df["interpretation_delta"].to_numpy(dtype=float), bins=30, color="#c44e52")
    axes[1].set_title("Violation Delta Histogram")
    axes[1].set_xlabel("interpretation_delta")
    axes[1].set_ylabel("count")

    fig.suptitle(
        f"{len(violation_df)} violation point(s) | log: {violation_log_path.name if violation_log_path else 'N/A'}",
        fontsize=12,
    )
    fig.tight_layout()

    summary_df = violation_df.groupby(["top_name", "bottom_name"], as_index=False).agg(
        point_count=("interpretation_delta", "size"),
        delta_min=("interpretation_delta", "min"),
        delta_max=("interpretation_delta", "max"),
        inline_min=("inline", "min"),
        inline_max=("inline", "max"),
        xline_min=("xline", "min"),
        xline_max=("xline", "max"),
    )
    display(summary_df)
